In [1]:
import sys, os
from dotenv import load_dotenv, find_dotenv
import pandas, httpx

load_dotenv(find_dotenv(usecwd=True))
print("Python:", sys.executable)  # deve contenere .venv
print("pandas:", pandas.__version__, "| httpx:", httpx.__version__)
for name in ["GRAPHDB_URL", "GRAPHDB_REPO_DEV", "LOCAL_MODEL_URL", "LOCAL_MODEL", "LOCAL_MODEL_API_KEY"]:
    print(name, "presente:", bool(os.getenv(name)))
print("GOOGLE_API_KEY (facoltativa in questa fase):", bool(os.getenv("GOOGLE_API_KEY")))


Python: /Users/aurorasancetta/Library/Mobile Documents/com~apple~CloudDocs/uni/magistrale/esami da dare/graph databases/project/progettoGDB/.venv/bin/python
pandas: 3.1.0 | httpx: 0.28.1
GRAPHDB_URL presente: True
GRAPHDB_REPO_DEV presente: True
LOCAL_MODEL_URL presente: True
LOCAL_MODEL presente: True
LOCAL_MODEL_API_KEY presente: True
GOOGLE_API_KEY (facoltativa in questa fase): False


In [2]:
import os, httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
endpoint = (
    os.environ["GRAPHDB_URL"].rstrip("/") + "/repositories/" +
    os.environ["GRAPHDB_REPO_DEV"]
)
query = """PREFIX ex: <http://example.org/semop/>
SELECT (COUNT(DISTINCT ?film) AS ?count)
WHERE { ?film ex:inSet ex:set-main . }"""

r = httpx.post(endpoint, data={"query": query},
               headers={"Accept": "application/sparql-results+json"}, timeout=30)
r.raise_for_status()
count = int(r.json()["results"]["bindings"][0]["count"]["value"])
print("GraphDB:", endpoint)
print("Film nel main set (dev):", count)
assert count == 1000, f"Attesi 1000 film, trovati {count}"


GraphDB: http://localhost:7200/repositories/films-dev
Film nel main set (dev): 1000


In [3]:
import os, httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
URL = os.environ["LOCAL_MODEL_URL"].rstrip("/")
H = {"Authorization": f"Bearer {os.getenv('LOCAL_MODEL_API_KEY', 'ollama')}"}
r = httpx.get(URL + "/models", headers=H, timeout=15)
print("Server locale:", URL, "| HTTP:", r.status_code)
r.raise_for_status()
print(r.json())  # Ollama o Unsloth Studio, secondo il proprio .env


Server locale: http://localhost:11434/v1 | HTTP: 200
{'object': 'list', 'data': [{'id': 'qwen2.5:3b', 'object': 'model', 'created': 1794153987, 'owned_by': 'library'}]}


In [5]:
import os, json, httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
URL = os.environ["LOCAL_MODEL_URL"].rstrip("/")
MODEL = os.environ["LOCAL_MODEL"]
H = {"Authorization": f"Bearer {os.getenv('LOCAL_MODEL_API_KEY', 'ollama')}"}

casi = {
    "positivo (atteso true)": "Dunkirk follows Allied soldiers trapped on the beaches of France in 1940 during World War II.",
    "negativo (atteso false)": "A romantic comedy follows two people falling in love in modern-day Paris.",
}
for nome, testo in casi.items():
    body = {
        "model": MODEL,
        "temperature": 0,
        "max_tokens": 300,
        "stream": False,
        "messages": [
            {"role": "system", "content":
                'Decide whether the input satisfies the criterion. Reply only with JSON containing a boolean field named "answer".'},
            {"role": "user", "content":
                f"Criterion: the film deals with the Second World War\nInput: {testo}"}
        ],
        "response_format": {"type": "json_object"},
    }
    r = httpx.post(URL + "/chat/completions", json=body, headers=H, timeout=120)
    print(nome, "-> HTTP", r.status_code)
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]
    print("Risposta:", content)
    answer = json.loads(content)["answer"]
    if not isinstance(answer, bool):
        raise ValueError("Il campo 'answer' deve essere booleano")
    print("Decisione:", answer)


positivo (atteso true) -> HTTP 200
Risposta: {
  "answer": true
}
Decisione: True
negativo (atteso false) -> HTTP 200
Risposta: {
  "answer": false
}
Decisione: False


In [6]:
import os, httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
key = os.getenv("GOOGLE_API_KEY")
if not key or key == "la_tua_chiave":
    print("Test Google saltato: GOOGLE_API_KEY non configurata (facoltativa).")
else:
    r = httpx.get("https://generativelanguage.googleapis.com/v1beta/models",
                  headers={"x-goog-api-key": key}, timeout=30)
    print(r.status_code)
    print([m["name"] for m in r.json().get("models", [])][:15] if r.status_code == 200 else r.text[:300])


Test Google saltato: GOOGLE_API_KEY non configurata (facoltativa).


In [7]:
import os, httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
key = os.getenv("GOOGLE_API_KEY")
if not key or key == "la_tua_chiave":
    print("Elenco modelli Google saltato: GOOGLE_API_KEY non configurata.")
else:
    r = httpx.get("https://generativelanguage.googleapis.com/v1beta/models?pageSize=200",
                  headers={"x-goog-api-key": key}, timeout=30)
    r.raise_for_status()
    for m in r.json().get("models", []):
        if "flash" in m["name"] and "generateContent" in m.get("supportedGenerationMethods", []):
            print(m["name"])


Elenco modelli Google saltato: GOOGLE_API_KEY non configurata.


In [8]:
import os, httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
key = os.getenv("GOOGLE_API_KEY")
MODEL = "gemini-3.1-flash-lite"

# Il test seguente può avere un costo: abilitarlo solo quando serve.
ESEGUI_TEST_GOOGLE_A_PAGAMENTO = False
if not ESEGUI_TEST_GOOGLE_A_PAGAMENTO:
    print("Test di generazione Google saltato (impostare True per eseguire una chiamata a pagamento).")
elif not key or key == "la_tua_chiave":
    print("Test Google non eseguito: manca una GOOGLE_API_KEY valida.")
else:
    body = {"model": MODEL, "temperature": 0, "max_tokens": 300,
            "messages": [{"role": "user", "content":
                "Criterion: the film deals with the Second World War\n"
                "Input: Dunkirk follows Allied soldiers trapped on the beaches of France in 1940 during World War II."}],
            "response_format": {"type": "json_schema", "json_schema": {
                "name": "answer", "schema": {"type": "object",
                "properties": {"answer": {"type": "boolean"}}, "required": ["answer"]}}}}
    r = httpx.post("https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
                   json=body, headers={"Authorization": f"Bearer {key}"}, timeout=60)
    print(r.status_code)
    print(r.json()["choices"][0]["message"]["content"] if r.status_code == 200 else r.text[:300])


Test di generazione Google saltato (impostare True per eseguire una chiamata a pagamento).
